In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

eps = 1e-6

# =========================
# 1. Load training datasets
# =========================
water_quality = pd.read_csv("../../data/water_quality_training_dataset.csv")
landsat = pd.read_csv("../../data/landsat_features_training.csv")
terraclimate = pd.read_csv("../../data/terraclimate_features_training.csv")

# =========================
# 2. Convert dates
# =========================
water_quality["Sample Date"] = pd.to_datetime(water_quality["Sample Date"], dayfirst=True)
landsat["Sample Date"] = pd.to_datetime(landsat["Sample Date"], dayfirst=True)
terraclimate["Sample Date"] = pd.to_datetime(terraclimate["Sample Date"], dayfirst=True)

# =========================
# 3. Create temporal features
# =========================
water_quality["month"] = water_quality["Sample Date"].dt.month
water_quality["year"] = water_quality["Sample Date"].dt.year
water_quality["dayofyear"] = water_quality["Sample Date"].dt.dayofyear

# Real seasonality
water_quality["sin_doy"] = np.sin(2 * np.pi * water_quality["dayofyear"] / 365.0)
water_quality["cos_doy"] = np.cos(2 * np.pi * water_quality["dayofyear"] / 365.0)

# =========================
# 4. Merge training datasets
# =========================
df = water_quality.merge(
    landsat,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df = df.merge(
    terraclimate,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 5. Feature engineering
# =========================
# Core v4.0 features
df["nir_swir16_ratio"] = df["nir"] / (df["swir16"] + eps)
df["nir_swir22_ratio"] = df["nir"] / (df["swir22"] + eps)
df["green_nir_ratio"] = df["green"] / (df["nir"] + eps)

df["nir_minus_swir16"] = df["nir"] - df["swir16"]
df["nir_minus_green"] = df["nir"] - df["green"]

df["ndmi_pet"] = df["NDMI"] * df["pet"]
df["mndwi_pet"] = df["MNDWI"] * df["pet"]

df["swir_ratio"] = df["swir16"] / (df["swir22"] + eps)

df["nir_pet"] = df["nir"] * df["pet"]
df["swir16_pet"] = df["swir16"] * df["pet"]
df["ndmi_day"] = df["NDMI"] * df["dayofyear"]

# New v4.5 features
df["ndmi_mndwi"] = df["NDMI"] * df["MNDWI"]
df["water_index"] = (df["green"] - df["swir16"]) / (df["green"] + df["swir16"] + eps)

# Defensive cleaning
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# =========================
# 6. Handle missing values
# =========================
train_medians = df.median(numeric_only=True)
df.fillna(train_medians, inplace=True)

# =========================
# 7. Define training features and targets
# =========================
targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

X = df.drop(columns=targets + ["Sample Date", "Latitude", "Longitude"])
y = df[targets]

X_train_medians = X.median(numeric_only=True)

# =========================
# 8. Train final model
# =========================
rf_final = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X, y)

# =========================
# 9. Load submission datasets
# =========================
submission = pd.read_csv("../../data/submission_template.csv")
landsat_val = pd.read_csv("../../data/landsat_features_validation.csv")
terraclimate_val = pd.read_csv("../../data/terraclimate_features_validation.csv")

# =========================
# 10. Convert dates
# =========================
submission["Sample Date"] = pd.to_datetime(submission["Sample Date"], dayfirst=True)
landsat_val["Sample Date"] = pd.to_datetime(landsat_val["Sample Date"], dayfirst=True)
terraclimate_val["Sample Date"] = pd.to_datetime(terraclimate_val["Sample Date"], dayfirst=True)

# =========================
# 11. Create temporal features
# =========================
submission["month"] = submission["Sample Date"].dt.month
submission["year"] = submission["Sample Date"].dt.year
submission["dayofyear"] = submission["Sample Date"].dt.dayofyear

# Real seasonality
submission["sin_doy"] = np.sin(2 * np.pi * submission["dayofyear"] / 365.0)
submission["cos_doy"] = np.cos(2 * np.pi * submission["dayofyear"] / 365.0)

# =========================
# 12. Merge validation datasets
# =========================
df_val = submission.merge(
    landsat_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df_val = df_val.merge(
    terraclimate_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 13. Same feature engineering
# =========================
# Core v4.0 features
df_val["nir_swir16_ratio"] = df_val["nir"] / (df_val["swir16"] + eps)
df_val["nir_swir22_ratio"] = df_val["nir"] / (df_val["swir22"] + eps)
df_val["green_nir_ratio"] = df_val["green"] / (df_val["nir"] + eps)

df_val["nir_minus_swir16"] = df_val["nir"] - df_val["swir16"]
df_val["nir_minus_green"] = df_val["nir"] - df_val["green"]

df_val["ndmi_pet"] = df_val["NDMI"] * df_val["pet"]
df_val["mndwi_pet"] = df_val["MNDWI"] * df_val["pet"]

df_val["swir_ratio"] = df_val["swir16"] / (df_val["swir22"] + eps)

df_val["nir_pet"] = df_val["nir"] * df_val["pet"]
df_val["swir16_pet"] = df_val["swir16"] * df_val["pet"]
df_val["ndmi_day"] = df_val["NDMI"] * df_val["dayofyear"]

# New v4.5 features
df_val["ndmi_mndwi"] = df_val["NDMI"] * df_val["MNDWI"]
df_val["water_index"] = (df_val["green"] - df_val["swir16"]) / (df_val["green"] + df_val["swir16"] + eps)

df_val.replace([np.inf, -np.inf], np.nan, inplace=True)

# =========================
# 14. Handle missing values
# =========================
df_val.fillna(train_medians, inplace=True)

# =========================
# 15. Prepare validation features
# =========================
X_val = df_val.drop(
    columns=[
        "Sample Date",
        "Latitude",
        "Longitude",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ],
    errors="ignore"
)

# Align columns exactly with training
X_val = X_val.reindex(columns=X.columns)
X_val = X_val.fillna(X_train_medians)

# =========================
# 16. Predict
# =========================
predictions = rf_final.predict(X_val)

# =========================
# 17. Build submission
# =========================
submission["Total Alkalinity"] = predictions[:, 0]
submission["Electrical Conductance"] = predictions[:, 1]
submission["Dissolved Reactive Phosphorus"] = predictions[:, 2]

submission_v4_5 = submission[
    [
        "Longitude",
        "Latitude",
        "Sample Date",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ]
]

# =========================
# 18. Export
# =========================
submission_v4_5.to_csv("../../submissions/submission_v4.5.csv", index=False)

# =========================
# 19. Quick check
# =========================
print(submission_v4_5.shape)
print(submission_v4_5.head())
print(submission_v4_5.isna().sum())

(200, 6)
   Longitude   Latitude Sample Date  Total Alkalinity  Electrical Conductance  \
0  27.822778 -32.043333  2014-09-01        132.457054               512.42286   
1  26.077500 -33.329167  2015-09-16        165.422056               541.02940   
2  27.640028 -32.991639  2015-05-07         76.886060               443.37602   
3  24.439167 -34.096389  2012-02-07         43.412459               111.58260   
4  28.581667 -32.000556  2014-10-01        106.622639               379.01615   

   Dissolved Reactive Phosphorus  
0                         38.814  
1                         44.360  
2                         29.436  
3                         10.490  
4                         23.999  
Longitude                        0
Latitude                         0
Sample Date                      0
Total Alkalinity                 0
Electrical Conductance           0
Dissolved Reactive Phosphorus    0
dtype: int64
